# LUMEN-299 · 3 — Mount Wilson, night by night

**The failure.** A season of sixteen hundred readings, four mirrors tried in turn, the
running mean checked after every night, and the baseline re-surveyed part way through.
The number reported at the end either has a plan behind it or it does not, and six months
later nobody remembers which — and nobody remembers that the mean was read forty times
before it was read once.

This notebook runs the 1926 season as a programme step: the plan filed and frozen before
the first reading, the nights assigned to mirrors by a rule that re-derives, the running
mean read with an interval that is valid however often it is read, the re-survey filed as
a departure from the plan, and the readout attached to the plan it answers.

| | reaches into |
|---|---|
| the plan, filed and frozen | `build.ExperimentBuilder`, `io.Catalog`, `io.Program`, `io.ExperimentRun` |
| which nights got which mirror | `design.assign`, `design.ArmAllocation` |
| reading the mean after every night | `design.confidence_sequence`, `design.AnytimeLook` |
| a departure from the plan | `io.Deviation`, `ExperimentRun.deviate`, `ExperimentRun.conformance` |
| the readout, and the one that filed nothing | `calibrate.Measurement`, `build.readouts_across` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import pandas as pd

import lightspeed as L

import tempfile
from pathlib import Path

from axiom.build import ExperimentBuilder, readouts_across
from axiom.calibrate import Measurement
from axiom.core import TimeWindow
from axiom.design import ConfidenceSequence, confidence_sequence
from axiom.io import Catalog, ExperimentRun, Program

from axiom.display import enable, show, table

sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, CRITICAL, ORANGE, band, caption, figure, lines, mark_x, mark_y, points

enable();  # every axiom result renders itself from here on

catalog = Catalog(Path(tempfile.mkdtemp()) / "lumen")
michelson = Program(party="rotating-mirror", program="michelson", started="1926-06-01",
                    description="the speed of light between Mount Wilson and Mount San Antonio")
catalog.register(michelson)
builder = ExperimentBuilder(catalog, michelson)
c_vacuo = L.estimand()
print(builder)

## 1 · The plan, before the first reading

Four roles make the plan: the apparatus as an expression tree, the surveyed baseline, the
calendar of nights, and which mirror each night runs. The last is an assignment — forty
nights into four arms in permuted blocks, so every mirror gets ten nights and the rule
that decided it travels with the plan and can be re-derived by anybody with the seed.
`commit` freezes the four together as one hash.

In [ ]:
nights = L.nights()
assigned = L.assign_nights()   # design.assign: four arms, permuted blocks, seed 1926
configuration_of_night = tuple(assigned.arm(n) for n in nights)
print("the rule:", assigned.spec.method, "| arms:", assigned.spec.allocation.arms, "| seed:", assigned.spec.seed)
print("nights per mirror:", assigned.counts(), "| the rule re-derives:", assigned.verify().status)

run = builder.plan(
    "MW-1926", estimand=c_vacuo,
    roles={"design": L.model(), "survey": L.MOUNT_WILSON,
           "season": TimeWindow(start=0, stop=L.NIGHTS, basis="per_period"),
           "mirrors": assigned.spec},
    at="1926-05-20T00:00:00+00:00",
)
run = builder.commit(run, at="1926-06-01T00:00:00+00:00", note="survey party off the mountain")
run = builder.start(run, at="1926-06-15T00:00:00+00:00")
print(run.summary())
table([[e.label, e.type_name.split(":")[-1], e.digest[:12] + "…"]
       for e in builder.store.find(tags={"experiment": "MW-1926"})],
      headers=("filed as", "type", "digest"), title="what the plan is made of")

## 2 · The season

Sixteen hundred readings, forty a night, each one the tree of notebook 2 plus the blur of
thirty-five kilometres of air. Every reading is reduced to a speed by reading the tree
backwards, and the first thing to check is that reading it backwards is reading it.

In [ ]:
season = L.mount_wilson(configuration_of_night=configuration_of_night)
speeds = L.season_speeds(season, L.MOUNT_WILSON)
season["c"] = speeds

# The inverse is the same tree: push each implied speed back through forward().
mirror = L.surface()
back = mirror.forward({"omega": season.omega.to_numpy(),
                       "null_angle": 4 * math.pi / season.faces.to_numpy()},
                      {"log_c": np.log(speeds), "log_D": np.full(len(season), math.log(L.MOUNT_WILSON.length_km))})
assert np.allclose(back, season.delta.to_numpy(), atol=1e-12)
print(f"{len(season):,} readings over {season.night.nunique()} nights;",
      "forward() reproduces every observed deflection from its implied speed")

by_night = season.groupby("night").agg(configuration=("configuration", "first"),
                                        c_mean=("c", "mean"), sd=("c", "std"), n=("c", "size"))
by_night["se"] = by_night.sd / np.sqrt(by_night.n)
table([[int(i), r.configuration, f"{r.c_mean:,.1f}", f"{r.se:,.1f}"]
       for i, r in by_night.head(6).iterrows()],
      headers=("night", "mirror", "mean c (km/s)", "se"), title="the first six nights")
by_mirror = season.groupby("configuration").c.agg(["mean", "std", "size"])
table([[m, f"{r['mean']:,.2f}", f"{r['std'] / math.sqrt(r['size']):,.2f}"]
       for m, r in by_mirror.iterrows()],
      headers=("mirror", "mean c", "se"), title="by mirror, all nights")

In [ ]:
groups = {m: (by_night.index[by_night.configuration == m].to_numpy(),
              by_night["c_mean"][by_night.configuration == m].to_numpy())
          for m in L.CONFIGURATIONS}
fig = points(
    groups,
    title="Forty nights, four mirrors",
    subtitle="the mean speed each night's forty readings imply, km/s",
    x_title="night", y_title="c (km/s)",
)
mark_y(fig, L.C_TRUE, text="the truth")
mark_y(fig, L.PRELIMINARY_1924, text="the 1924 preliminary value", dash="dash")
caption(fig, "No night is far from any other and no mirror stands apart; the spread from "
             "night to night is the refraction and the fork, not the mirror. Every night "
             "sits below the preliminary value the first season had published.")

## 3 · Reading the mean after every night

Michelson did not wait for the season to end before looking. The question he was asking
each night was whether the running mean had moved off the **299,802 km/s** he had
published from the first Mount Wilson season in 1924 — and a fixed-sample interval read
forty times is not a 95% interval. `confidence_sequence` takes the same running statistic and returns
an interval valid at every look at once, labelled `anytime` so it cannot be mistaken for
the other kind, with the e-value that makes it so.

In [ ]:
NULL = L.PRELIMINARY_1924
cumulative = season.groupby("night").c.agg(["sum", "size"]).cumsum()
running_mean = (cumulative["sum"] / cumulative["size"]).to_numpy()
running_se = np.array([season.c[season.night <= k].std(ddof=1) / math.sqrt(cumulative["size"].iloc[k])
                       for k in range(L.NIGHTS)])
z = (running_mean - NULL) / running_se
information = (np.arange(L.NIGHTS) + 1) / L.NIGHTS
seq: ConfidenceSequence = confidence_sequence(
    z, information, alpha=0.05, name="MW-1926 running mean", labels=list(nights))
se_full = running_se[-1]
looks = [0, 1, 4, 9, 19, 39]
table([[seq.looks[k].label, f"{running_mean[k]:,.1f}",
        f"[{NULL + seq.looks[k].interval.lower * se_full:,.1f}, {NULL + seq.looks[k].interval.upper * se_full:,.1f}]",
        f"{seq.looks[k].evalue:.3g}", str(seq.looks[k].crossed)] for k in looks],
      headers=("after", "running mean", "anytime 95% interval (km/s)", "e-value", "excludes 299,802"))
newcomb_out = next(k for k, l in enumerate(seq.looks) if NULL + l.interval.upper * se_full < 299_860.0)
print(f"\nthe 1924 value is excluded, validly, from night {seq.crossed_at + 1} on;")
print(f"Newcomb's 299,860 is outside the anytime interval from night {newcomb_out + 1};")
print(f"the final anytime interval is {seq.width_ratio():.2f}x the width of a fixed-sample one read once")

In [ ]:
x = np.arange(1, L.NIGHTS + 1)
fig = figure(
    title="The running mean, and the interval it is entitled to",
    subtitle="the anytime interval (shaded) against the fixed-sample Wald interval nobody may read forty times",
    x_title="nights into the season", y_title="c (km/s)",
)
band(fig, x, NULL + np.array([l.interval.lower for l in seq.looks]) * se_full,
     NULL + np.array([l.interval.upper for l in seq.looks]) * se_full, color=BLUE, name="anytime 95%")
fig.add_scatter(x=x, y=running_mean - 1.96 * running_se, mode="lines",
                line={"color": ORANGE, "dash": "dot"}, name="fixed-sample 95%", showlegend=True)
fig.add_scatter(x=x, y=running_mean + 1.96 * running_se, mode="lines",
                line={"color": ORANGE, "dash": "dot"}, showlegend=False)
fig.add_scatter(x=x, y=running_mean, mode="lines", line={"color": BLUE, "width": 2},
                name="running mean", showlegend=True)
mark_y(fig, L.C_TRUE, text="the truth")
mark_y(fig, NULL, text="the 1924 preliminary", dash="dash")
fig.update_yaxes(range=[L.C_TRUE - 25, NULL + 15])
caption(fig, "The shaded interval is wider at every night than the dotted one, by about half "
             "again, and that gap is the price of looking every night. Both leave the 1924 "
             "value behind within the first ten nights; only one of them was allowed to say so.")

## 4 · The baseline was re-surveyed

In the middle of the season the Coast and Geodetic Survey re-measured the line. It moved
by five centimetres in thirty-five kilometres, and every speed in the season moves with
it. That is a change to a committed role, and the run records it as a `Deviation` — what
changed, from what, to what, why — rather than quietly replacing the number. The plan's
hash no longer matches the roles, and the object says so.

In [ ]:
before = L.MOUNT_WILSON.length_km
after = L.MOUNT_WILSON_RESURVEY.length_km
shift = L.C_TRUE * (after / before - 1.0)
print(f"baseline {before:.5f} -> {after:.5f} km: every speed moves by {shift:+.2f} km/s")

run = run.deviate("survey", L.MOUNT_WILSON_RESURVEY,
                  reason="the Coast and Geodetic Survey re-measured the line in August",
                  state="asserted", at="1926-08-10T00:00:00+00:00")
speeds = L.season_speeds(season, L.MOUNT_WILSON_RESURVEY)
season["c"] = speeds
print("plan hash still", run.plan_hash[:12] + "…", "| roles now hash to", run.plan_digest()[:12] + "…")
print("conformance while still running:", run.conformance().status, "—", run.conformance().reason)

## 5 · The readout

The season's mean, with a standard error that is a budget rather than a scatter: the
statistical error of 1,600 readings, the calibration of the tuning fork that timed the
mirror, and the survey. The fork is the largest term and no reading can see it, which is
why the published ±4 was two and a half times the scatter — and why the number that goes
into the corpus carries the budget, not the scatter.

In [ ]:
statistical = float(speeds.std(ddof=1) / math.sqrt(speeds.size))
fork = L.OMEGA_CALIBRATION * L.C_TRUE
survey = L.MOUNT_WILSON_RESURVEY.relative_error * L.C_TRUE
budget = math.sqrt(statistical**2 + fork**2 + survey**2)
table([["1,600 readings", f"{statistical:.2f}"], ["tuning fork, 1 in 10^5", f"{fork:.2f}"],
       ["survey, 1 in 7 million", f"{survey:.3f}"], ["in quadrature", f"{budget:.2f}"]],
      headers=("term", "km/s"), title="the error budget")

measurement = Measurement(
    estimand=c_vacuo, estimate=float(speeds.mean()), se=budget, definition="wald", mass=0.95,
    method="rotating_mirror", n_units=L.NIGHTS, n_periods=L.READINGS_PER_NIGHT, source="MW-1926",
)
run = builder.read(run, measurement, at="1926-09-30T00:00:00+00:00")
print(f"MW-1926: c = {measurement.estimate:,.1f} ± {measurement.se:.1f} km/s "
      f"({measurement.interval.text()})  truth {L.C_TRUE:,.1f}")
print(run.summary())
show(run.conformance())

world = L.simulate()
assert abs(world["michelson_1926"].read - measurement.estimate) < 1e-9
print("\nthe corpus in notebook 4 holds exactly this number for 1926")

`downgraded`, with the re-survey riding on the verdict as an assumption in the state it
was filed under. The readout is still the quantity the plan committed to — a re-survey
does not change what was measured — and somebody asserted so at the time. That is the
difference between a number with a departure recorded and a number with a departure
remembered.

## 6 · The one that changed its plan and filed nothing

Nine years later the same laboratory ran the mile-long vacuum tube. Suppose its plan was
committed with one survey and read with another, and nobody filed the change — which is
the ordinary way a plan changes. `conformance` does not say `downgraded`; it says
`blocked`, and `readouts_across` leaves the run out of anything a programme decides on.

In [ ]:
pease = Program(party="rotating-mirror", program="pease-pearson", started="1935-01-01")
catalog.register(pease)
tube = ExperimentBuilder(catalog, pease)
tube_survey = L.BaselineSurvey(length_km=8.0, relative_error=2e-6, surveyed_by="tape, in the tube", year=1935)
run35 = tube.plan("MPP-1935", estimand=c_vacuo,
                  roles={"design": L.model(), "survey": tube_survey,
                         "season": TimeWindow(start=0, stop=120, basis="per_period")},
                  at="1935-01-15T00:00:00+00:00")
run35 = tube.start(tube.commit(run35, at="1935-02-01T00:00:00+00:00"), at="1935-03-01T00:00:00+00:00")

resurveyed = tube_survey.model_copy(update={"length_km": 8.0004})
tampered = run35.model_copy(update={"roles": {**run35.roles, "survey": resurveyed.content_hash()}})
r35 = world["pease_pearson_1935"]
run35 = tube.read(tampered, Measurement(estimand=c_vacuo, estimate=r35.read, se=r35.se,
                                        method="rotating_mirror_vacuum", source="MPP-1935"),
                  at="1935-12-01T00:00:00+00:00")
print(run35.summary())

table([[r.experiment, r.scope, r.stage, r.conformance().status] for r in (*builder.runs(), *tube.runs())],
      headers=("experiment", "scope", "stage", "conformance"), title="every run filed, latest version")
print("readouts a programme would take:", [r.key for r in readouts_across([builder, tube])])

## What notebook 3 established

1. **The plan was frozen before the first reading**, as four typed roles under one hash,
   and the nights went to mirrors by a rule that anybody with the seed can re-derive.
2. **Reading the mean every night is priced.** The anytime interval is about half again
   wider than the fixed-sample one at every look, excludes the 1924 preliminary value
   validly from the ninth night, and carries an e-value the programme can use later.
3. **The re-survey is a deviation, not a correction.** It moved every speed by four tenths
   of a kilometre a second, and the run says `downgraded` with the departure as an
   assumption rather than `identified` with a number nobody can explain.
4. **The error is a budget.** The fork outweighs the scatter more than two to one and no
   reading can see it; the corpus gets the budget.
5. **An unfiled change is `blocked`**, and a blocked run is not evidence. The vacuum tube's
   number is filed, findable, and excluded from every decision the programme takes until
   somebody says what changed.

Notebook 4 pools the century.